# Module 9 · Lab 9.1 — MCP & A2A Protocol Deep Dive
## Open standards for agent interop — one banking MCP server over stdio and HTTP, and A2A Agent Cards you can discover

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---

Two protocols are reshaping multi-system AI: **MCP** (Model Context Protocol — agent ↔ tools) and **A2A** (Agent-to-Agent — agent ↔ agent). Both are open standards, both have major industry backing. This lab teaches both from scratch.

## Business Scenario: Why Protocols Matter

**The Pre-Protocol World:** Every team writes custom integration code. ShopSmart wraps its inventory API one way, the banking team another, the insurance team a third. Result: 100+ bespoke integrations, no reuse, no portability.

**The Protocol World:** Standards. Just like USB-C lets any laptop charge from any port, MCP lets any agent talk to any tool. A2A lets any agent collaborate with any other agent.

1. **Tool Integration Chaos:** Without MCP, every agent ↔ tool integration is custom. SecureBank's analyst built 23 different OpenAI function-calling wrappers in 18 months — each subtly different.

2. **Agent Silos:** Without A2A, agents from different vendors can't collaborate. Salesforce Einstein can't easily delegate to a SAP agent. They speak different dialects.

3. **Adoption is Real (April 2026):** 150+ organizations support MCP. A2A launched April 2025, now stewarded by Linux Foundation. These are not hypotheticals — they're production realities.

**Our Solution:** Build a **Banking MCP server** and connect to it three ways — in memory, over **stdio** and over **HTTP**. Define **A2A Agent Cards** for Insurance + Compliance agents. Show how the two protocols complement each other.

**Learning objectives.** By the end of this lab you can:
1. Explain the N×M integration problem MCP solves, and MCP's three primitives — tools, resources, prompts.
2. Write an MCP server with FastMCP and reach it in memory, over **stdio** and over **Streamable HTTP**
   with the *same* client code.
3. Hand MCP tools to a LangChain agent with `langchain-mcp-adapters` — and withhold the ones it must not call.
4. Publish and discover an **A2A Agent Card**, and say when a problem needs MCP, A2A or neither.

**Keys:** only the trainer-supplied `OPENAI_API_KEY` (for the agent in §9).

> 🗺️ **Workshop run-list.**
> **Live in session:** §1–12 — what MCP is, the server as a real program, in memory, calls / resources / prompts, stdio ⭐, the same server over HTTP ⭐, an agent with read-only MCP tools, A2A cards published and discovered, when to use which, stopping the server
> **After the session:** §13 try on your own — kept in full so you can finish at your own pace (marked ⏭️).

## 1. Set-up

In [1]:
# 📦 Packages this notebook needs. On the workshop VM they are pre-installed from
# the workshop's requirements.txt, so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU fastmcp mcp httpx langchain langchain-openai langchain-mcp-adapters python-dotenv

from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_PACKAGES = [
    "fastmcp",
    "mcp",
    "httpx",
    "langchain",
    "langchain-openai",
    "langchain-mcp-adapters",
    "python-dotenv",
]
for package_name in NOTEBOOK_PACKAGES:
    try:
        print(f"{package_name:<26} {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name:<26} NOT INSTALLED — uncomment the !pip line above")

fastmcp                    3.4.7
mcp                        1.28.1
httpx                      0.28.1
langchain                  1.4.3
langchain-openai           1.6.6
langchain-mcp-adapters     0.3.2
python-dotenv              1.2.3


In [2]:
import os

# Where to create each key. The trainer supplies OPENAI_API_KEY on the workshop VM;
# you create any other key yourself and add it to the workshop's .env file as KEY_NAME=value.
KEY_SIGNUP_PAGES = {
    "OPENAI_API_KEY": "supplied on the workshop VM (elsewhere: https://platform.openai.com/api-keys)",
    "TAVILY_API_KEY": "https://app.tavily.com (free tier)",
    "GROQ_API_KEY": "https://console.groq.com/keys (free tier)",
    "WEATHER_API_KEY": "https://www.weatherapi.com/signup.aspx (free tier)",
    "LANGSMITH_API_KEY": "https://smith.langchain.com -> Settings -> API Keys (free tier)",
    "ANTHROPIC_API_KEY": "https://console.anthropic.com/settings/keys",
    "GEMINI_API_KEY": "https://aistudio.google.com/app/apikey",
}


def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere).

    Prints one line per key, then stops with instructions if a required key is missing.

    Args:
        required: the environment-variable names this lab needs.
    """
    try:
        from google.colab import userdata          # running on Google Colab
        for key_name in required:
            try:
                os.environ[key_name] = userdata.get(key_name)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # the workshop VM, or any local machine
        from dotenv import find_dotenv, load_dotenv
        load_dotenv(find_dotenv(usecwd=True))        # searches upward for the workshop's .env file
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for key_name in required:
        print(f"   {key_name:<22} {'✅' if os.environ.get(key_name) else '❌ MISSING'}")
    missing_keys = [key_name for key_name in required if not os.environ.get(key_name)]
    if missing_keys:
        instructions = "\n".join(
            f"  {key_name}: {KEY_SIGNUP_PAGES.get(key_name, 'see README.md in the workshop folder')}"
            for key_name in missing_keys)
        raise RuntimeError(
            "This lab needs API keys that are not set yet. Create each one, add it to\n"
            "the workshop's .env file as KEY_NAME=value, then re-run this cell:\n" + instructions)


load_keys(["OPENAI_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅


In [3]:
import asyncio
import json
import subprocess
import sys
import time

import httpx
from fastmcp import Client

# Jupyter already runs an event loop and supports top-level `await` and `async with`, so every
# async call below runs directly. No asyncio.run(), no nest_asyncio — both break inside a notebook.
print("Imports ready.")

Imports ready.


## 2. What Is MCP? (Model Context Protocol)

**MCP is to AI agents what USB-C is to laptops.** A standard protocol that lets any agent talk to any tool, data source, or workflow — without custom integration code per pairing.

Three primitives:
- **Tools:** Functions an LLM can call (e.g., `check_balance(account_id)`)
- **Resources:** Data the LLM can read (e.g., `policy://banking-terms`)
- **Prompts:** Reusable prompt templates (e.g., `transfer_confirmation`)

**Architecture:**

```
[MCP Server]              [MCP Client]
 - Tools                   - Connects via stdio/HTTP/SSE
 - Resources    <-->       - Lists available capabilities
 - Prompts                 - Calls tools, reads resources
```

### MCP vs Custom Tool Integration

| Aspect | Custom Integration | MCP |
|--------|-------------------|-----|
| **Reusability** | One-off per agent | Server reused across all clients |
| **Discoverability** | Manual docs | Auto-discoverable via `list_tools()` |
| **Cross-framework** | Lock-in | Works with LangChain, OpenAI SDK, AutoGen, Claude Desktop |
| **Maintenance** | Update N agents per change | Update 1 server, all clients benefit |
| **Industry support** | DIY | 150+ orgs (April 2026) |

### One protocol, three ways to reach a server

| Transport | How the client reaches the server | Use it for |
|---|---|---|
| **In-memory** | `Client(server_object)` — same Python process, no I/O | unit tests; learning the API |
| **stdio** | `Client("banking_mcp_server.py")` — the client **starts the server as a child process** and talks over stdin/stdout | local tools: an IDE, a desktop assistant, a notebook |
| **Streamable HTTP** | `Client("http://127.0.0.1:8765/mcp")` — the server runs on its own and **listens on a port**; many clients connect | shared, deployed services |

The older HTTP+SSE transport is the one the ASCII diagram above still mentions; new servers use Streamable
HTTP. The point to carry away: **the client code does not change between transports** — only the target
you hand to `Client(...)`.

## 3. Build a Banking MCP Server

We'll create a SecureBank India MCP server with 3 tools, 1 resource, and 1 prompt template — written
to its **own file**, because stdio and HTTP both need a process to start. Three details matter:

- **No `print()` in the server.** Over stdio, *stdout is the protocol channel*: a stray print corrupts
  the JSON-RPC stream. (The startup banner is switched off for the same reason.)
- **The transport comes from the environment** — `TRANSPORT`, `HOST`, `PORT` — so one file serves stdio
  by default and HTTP when asked.
- **A `/health` route that proves the data is loaded,** not merely that the process is up — the lesson a
  real deployment teaches the hard way. It is an ordinary web route: MCP clients never see it.

In [4]:
%%writefile banking_mcp_server.py
"""SecureBank India MCP server: 3 tools, 1 resource, 1 prompt.

An MCP client starts it over stdio (the default). To serve it over the network instead:
    TRANSPORT=http HOST=127.0.0.1 PORT=8765 python banking_mcp_server.py
Over stdio, stdout IS the protocol channel, so this file never prints.
"""

import os

from fastmcp import FastMCP
from starlette.responses import PlainTextResponse

# --- Banking Data (Indian context) ---
ACCOUNTS_DATABASE = {
    "ACC001": {"name": "Rajesh Sharma", "balance": 245000.00, "city": "Mumbai", "type": "Savings"},
    "ACC002": {"name": "Priya Patel", "balance": 1850000.00, "city": "Ahmedabad", "type": "Current"},
    "ACC003": {"name": "Amit Verma", "balance": 78500.00, "city": "Delhi", "type": "Savings"},
}

BANKING_TERMS = """SecureBank India Terms & Conditions:
- Minimum balance: ₹10,000 (Savings), ₹50,000 (Current)
- IMPS transfer limit: ₹5,00,000 per transaction
- NEFT cut-off: 6 PM on working days
- RTGS minimum: ₹2,00,000
- Service charges: ₹18 + GST per transaction beyond free limit"""

# --- Build FastMCP Server ---
banking_server = FastMCP("SecureBankServer")


# Tool 1: Check balance
@banking_server.tool
def check_balance(account_id: str) -> str:
    """Check the balance of a SecureBank account."""
    account = ACCOUNTS_DATABASE.get(account_id.upper())
    if not account:
        return f"Account {account_id} not found."
    return f"Account {account_id} ({account['name']}, {account['type']}): ₹{account['balance']:,.2f}"


# Tool 2: Transfer funds
@banking_server.tool
def transfer_funds(from_account: str, to_account: str, amount: float) -> str:
    """Transfer funds between two SecureBank accounts."""
    source_account = ACCOUNTS_DATABASE.get(from_account.upper())
    destination_account = ACCOUNTS_DATABASE.get(to_account.upper())
    if not source_account or not destination_account:
        return "One or both accounts not found."
    if source_account["balance"] < amount:
        return f"Insufficient balance. Available: ₹{source_account['balance']:,.2f}"
    return f"Transfer of ₹{amount:,.2f} from {from_account} to {to_account} authorized."


# Tool 3: Account holder info
@banking_server.tool
def get_account_holder(account_id: str) -> str:
    """Get account holder name and city."""
    account = ACCOUNTS_DATABASE.get(account_id.upper())
    if not account:
        return f"Account {account_id} not found."
    return f"{account['name']} from {account['city']}"


# Resource: Banking terms
@banking_server.resource("policy://banking-terms")
def banking_terms_resource() -> str:
    """SecureBank India Terms & Conditions."""
    return BANKING_TERMS


# Prompt template: Transfer confirmation
@banking_server.prompt
def transfer_confirmation_prompt(amount: float, recipient: str) -> str:
    """Generate a transfer confirmation prompt for the user."""
    return (
        f"Please confirm: You wish to transfer ₹{amount:,.2f} to {recipient}. "
        "This action is irreversible. Reply 'CONFIRM' to proceed or 'CANCEL' to abort."
    )


# Operations route — served over HTTP only; MCP clients never see it.
@banking_server.custom_route("/health", methods=["GET"])
async def health_check(request):
    """Answer OK only when the account data is loaded — a running process is not proof of health."""
    if not ACCOUNTS_DATABASE:
        return PlainTextResponse("DEGRADED: no accounts loaded", status_code=503)
    return PlainTextResponse(f"OK - {len(ACCOUNTS_DATABASE)} accounts", status_code=200)


if __name__ == "__main__":
    transport = os.environ.get("TRANSPORT", "stdio")
    if transport == "stdio":
        banking_server.run(show_banner=False)
    else:   # "http" = Streamable HTTP, served at /mcp
        banking_server.run(transport=transport, host=os.environ.get("HOST", "127.0.0.1"),
                           port=int(os.environ.get("PORT", "8765")), show_banner=False)

Writing banking_mcp_server.py


## 4. Connect via MCP Client (In-Memory)

In production, MCP servers run as separate processes via stdio (subprocess) or HTTP — §6 and §7 do both. First we call the tools directly, in memory: hand `Client` the server *object* and FastMCP connects to it inside this Python process — same API surface, simpler runtime.

In [5]:
# --- Verify Server Configuration (Inspect Capabilities) ---
# In production, the server would run via stdio/HTTP. Here we inspect it in memory first.
import importlib

import banking_mcp_server

banking_mcp_server = importlib.reload(banking_mcp_server)   # a re-run picks up edits to the file
banking_server = banking_mcp_server.banking_server


async def list_capabilities(client_target):
    """List all tools, resources and prompts an MCP server offers, through one client session."""
    async with Client(client_target) as mcp_client:
        tools = await mcp_client.list_tools()
        resources = await mcp_client.list_resources()
        prompts = await mcp_client.list_prompts()
    return tools, resources, prompts


def print_capabilities(capabilities, transport_label):
    """Print the tools, resources and prompts returned by list_capabilities()."""
    tools_list, resources_list, prompts_list = capabilities
    print(f"MCP Server Capabilities (over {transport_label}):")
    print(f"\nTools ({len(tools_list)}):")
    for tool_info in tools_list:
        print(f"  - {tool_info.name}: {tool_info.description}")
    print(f"\nResources ({len(resources_list)}):")
    for resource_info in resources_list:
        print(f"  - {resource_info.uri}: {resource_info.description}")
    print(f"\nPrompts ({len(prompts_list)}):")
    for prompt_info in prompts_list:
        print(f"  - {prompt_info.name}: {prompt_info.description}")


print_capabilities(await list_capabilities(banking_server), "in memory")

MCP Server Capabilities (over in memory):

Tools (3):
  - check_balance: Check the balance of a SecureBank account.
  - transfer_funds: Transfer funds between two SecureBank accounts.
  - get_account_holder: Get account holder name and city.

Resources (1):
  - policy://banking-terms: SecureBank India Terms & Conditions.

Prompts (1):
  - transfer_confirmation_prompt: Generate a transfer confirmation prompt for the user.


## 5. Call tools, read a resource, render a prompt

`call_tool` returns a result object: `.data` is the deserialised return value, and `.content` holds the
raw MCP content blocks. One session serves all the calls.

In [6]:
async def run_banking_calls(client_target):
    """Call the three tools, read the terms resource and render the prompt, in one MCP session."""
    tool_calls = [
        ("check_balance", {"account_id": "ACC001"}),
        ("get_account_holder", {"account_id": "ACC002"}),
        ("transfer_funds", {"from_account": "ACC001", "to_account": "ACC003", "amount": 5000.0}),
    ]
    async with Client(client_target) as mcp_client:
        # --- Call MCP Tools ---
        print("MCP Tool Calls:")
        print("=" * 60)
        for tool_name, arguments in tool_calls:
            call_result = await mcp_client.call_tool(tool_name, arguments)
            print(f"{tool_name}({', '.join(str(value) for value in arguments.values())}):\n"
                  f"  -> {call_result.data}")
        # --- Read MCP Resource ---
        terms_contents = await mcp_client.read_resource("policy://banking-terms")
        prompt_result = await mcp_client.get_prompt("transfer_confirmation_prompt",
                                                    {"amount": 25000, "recipient": "ACC003"})
    print("\nResource: policy://banking-terms")
    print("=" * 60)
    print(terms_contents[0].text)
    print("\nPrompt: transfer_confirmation_prompt(amount=25000, recipient='ACC003')")
    print("=" * 60)
    print(prompt_result.messages[0].content.text)


await run_banking_calls(banking_server)

MCP Tool Calls:
check_balance(ACC001):
  -> Account ACC001 (Rajesh Sharma, Savings): ₹245,000.00
get_account_holder(ACC002):
  -> Priya Patel from Ahmedabad
transfer_funds(ACC001, ACC003, 5000.0):
  -> Transfer of ₹5,000.00 from ACC001 to ACC003 authorized.

Resource: policy://banking-terms
SecureBank India Terms & Conditions:
- Minimum balance: ₹10,000 (Savings), ₹50,000 (Current)
- IMPS transfer limit: ₹5,00,000 per transaction
- NEFT cut-off: 6 PM on working days
- RTGS minimum: ₹2,00,000
- Service charges: ₹18 + GST per transaction beyond free limit

Prompt: transfer_confirmation_prompt(amount=25000, recipient='ACC003')
Please confirm: You wish to transfer ₹25,000.00 to ACC003. This action is irreversible. Reply 'CONFIRM' to proceed or 'CANCEL' to abort.


> **A prompt template is not an LLM call.** `get_prompt` asks the *server* to fill in its template and
> hand the messages back; the client decides what to do with them. Tools act, resources inform, prompts
> guide — three different primitives, one protocol.

## 6. Connect over stdio — the client starts the server

Hand `Client` a stdio transport for the script and it launches that script as a child process (with
this notebook's own Python), speaks MCP over its stdin/stdout, and stops it when the `async with` block
ends. Entering the block performs the MCP `initialize` handshake; the `list_*` calls are how a client
**discovers** what a server offers — nothing is imported.

> **Notebook detail.** In a plain Python program, `Client("banking_mcp_server.py")` is all it takes. In a
> notebook, build the transport yourself and give it a `log_file`: the child process inherits the
> parent's stderr, and Jupyter's stderr is not a real file — without one, the connection fails with a
> terse `fileno` error.

In [7]:
from pathlib import Path

from fastmcp.client.transports import PythonStdioTransport

SERVER_SCRIPT = "banking_mcp_server.py"
STDIO_TRANSPORT = PythonStdioTransport(
    SERVER_SCRIPT,
    log_file=Path("banking_mcp_server_stdio.log"),   # the server's stderr goes here, not to Jupyter
    keep_alive=False,                                # one server process per session, stopped after
)

# The same two helpers as in memory — only the target changes.
print_capabilities(await list_capabilities(STDIO_TRANSPORT), "stdio")
print()
await run_banking_calls(STDIO_TRANSPORT)

MCP Server Capabilities (over stdio):

Tools (3):
  - check_balance: Check the balance of a SecureBank account.
  - transfer_funds: Transfer funds between two SecureBank accounts.
  - get_account_holder: Get account holder name and city.

Resources (1):
  - policy://banking-terms: SecureBank India Terms & Conditions.

Prompts (1):
  - transfer_confirmation_prompt: Generate a transfer confirmation prompt for the user.



MCP Tool Calls:
check_balance(ACC001):
  -> Account ACC001 (Rajesh Sharma, Savings): ₹245,000.00
get_account_holder(ACC002):
  -> Priya Patel from Ahmedabad
transfer_funds(ACC001, ACC003, 5000.0):
  -> Transfer of ₹5,000.00 from ACC001 to ACC003 authorized.



Resource: policy://banking-terms
SecureBank India Terms & Conditions:
- Minimum balance: ₹10,000 (Savings), ₹50,000 (Current)
- IMPS transfer limit: ₹5,00,000 per transaction
- NEFT cut-off: 6 PM on working days
- RTGS minimum: ₹2,00,000
- Service charges: ₹18 + GST per transaction beyond free limit

Prompt: transfer_confirmation_prompt(amount=25000, recipient='ACC003')
Please confirm: You wish to transfer ₹25,000.00 to ACC003. This action is irreversible. Reply 'CONFIRM' to proceed or 'CANCEL' to abort.


## 7. Serve it over HTTP — one server, many clients

stdio gives each client a private server process. A shared service instead runs **on its own** and
listens on a port. The cell below starts the same file as a background process with `TRANSPORT=http`.

- It passes an **argument list, not a shell command** — so the `&` in this module's folder name, which
  PowerShell and bash both treat as syntax, cannot break the launch.
- Then it waits for **readiness, not a timer**: it polls `/health` until the server answers, and keeps the
  last failure reason instead of a bare "timed out".
- If a server from an earlier run is already answering on the port, it reuses that one.

In [8]:
SERVER_PORT = 8765
SERVER_BASE_URL = f"http://127.0.0.1:{SERVER_PORT}"
MCP_ENDPOINT_URL = f"{SERVER_BASE_URL}/mcp"     # Streamable HTTP is served at /mcp
HEALTH_TIMEOUT_SECONDS = 20


async def wait_until_healthy(health_url=f"{SERVER_BASE_URL}/health", timeout=HEALTH_TIMEOUT_SECONDS):
    """Poll /health until it answers 200 OK or the timeout expires.

    Returns:
        (is_healthy, detail) — on failure `detail` is the LAST failure seen, because "it did not
        come up" is not a diagnosis.
    """
    started_at = time.time()
    last_failure = "no probe completed"
    while time.time() - started_at < timeout:
        try:
            async with httpx.AsyncClient(timeout=2.0) as probe_client:
                response = await probe_client.get(health_url)
            if response.status_code == 200:
                return True, response.text
            last_failure = f"HTTP {response.status_code} — {response.text.strip()[:150]}"
        except (httpx.ConnectError, httpx.ConnectTimeout) as connect_error:
            last_failure = f"cannot connect ({type(connect_error).__name__}) — is the server running?"
        await asyncio.sleep(0.5)
    return False, last_failure


already_running, _ = await wait_until_healthy(timeout=1)
if already_running:
    http_server_process = None
    print(f"A server is already answering on port {SERVER_PORT} — reusing it.")
else:
    server_environment = {**os.environ, "TRANSPORT": "http", "HOST": "127.0.0.1",
                          "PORT": str(SERVER_PORT)}
    server_log = open("banking_mcp_server.log", "w", encoding="utf-8")
    http_server_process = subprocess.Popen([sys.executable, SERVER_SCRIPT], env=server_environment,
                                           stdout=server_log, stderr=subprocess.STDOUT)

is_healthy, health_detail = await wait_until_healthy()
print(f"Server health: {'OK' if is_healthy else 'FAILED'}  ->  {health_detail}")
if not is_healthy:
    raise RuntimeError(f"No healthy server at {SERVER_BASE_URL}. Last probe: {health_detail}\n"
                       "Read banking_mcp_server.log next to this notebook for the server's own error.")

Server health: OK  ->  OK - 3 accounts


## 8. Same client code, different transport

Here is the whole point of a protocol. The two helper functions from §4 and §5 run **unchanged** — only
the target moves from a server object or a script path to a URL, and `Client` switches to Streamable HTTP.

In [9]:
print_capabilities(await list_capabilities(MCP_ENDPOINT_URL), "Streamable HTTP")
print()
await run_banking_calls(MCP_ENDPOINT_URL)

MCP Server Capabilities (over Streamable HTTP):

Tools (3):
  - check_balance: Check the balance of a SecureBank account.
  - transfer_funds: Transfer funds between two SecureBank accounts.
  - get_account_holder: Get account holder name and city.

Resources (1):
  - policy://banking-terms: SecureBank India Terms & Conditions.

Prompts (1):
  - transfer_confirmation_prompt: Generate a transfer confirmation prompt for the user.



MCP Tool Calls:
check_balance(ACC001):
  -> Account ACC001 (Rajesh Sharma, Savings): ₹245,000.00
get_account_holder(ACC002):
  -> Priya Patel from Ahmedabad
transfer_funds(ACC001, ACC003, 5000.0):
  -> Transfer of ₹5,000.00 from ACC001 to ACC003 authorized.

Resource: policy://banking-terms
SecureBank India Terms & Conditions:
- Minimum balance: ₹10,000 (Savings), ₹50,000 (Current)
- IMPS transfer limit: ₹5,00,000 per transaction
- NEFT cut-off: 6 PM on working days
- RTGS minimum: ₹2,00,000
- Service charges: ₹18 + GST per transaction beyond free limit

Prompt: transfer_confirmation_prompt(amount=25000, recipient='ACC003')
Please confirm: You wish to transfer ₹25,000.00 to ACC003. This action is irreversible. Reply 'CONFIRM' to proceed or 'CANCEL' to abort.


## 9. MCP + LLM Integration

Bridge MCP tools to LangChain so an LLM agent can call them via standard tool-calling. This is how MCP becomes useful in real applications — the LLM "discovers" available tools via MCP and invokes them through the standard interface.

In production, you'd use `langchain-mcp-adapters` — so this lab does. `load_mcp_tools` turns every tool the
session discovers into a LangChain tool that calls back over MCP. The agent must run **while the session is
open**, because that is the connection the tools use.

**Withhold what the agent must not call.** The server offers `transfer_funds`; this assistant only answers
questions, so it gets the two read-only tools and nothing that moves money.

In [10]:
# --- Convert MCP Tools to LangChain Tools ---
from langchain.agents import create_agent
from langchain_mcp_adapters.tools import load_mcp_tools

READ_ONLY_TOOL_NAMES = {"check_balance", "get_account_holder"}   # transfer_funds is withheld

async with Client(MCP_ENDPOINT_URL) as mcp_client:
    discovered_tools = await load_mcp_tools(mcp_client.session)
    read_only_tools = [mcp_tool for mcp_tool in discovered_tools if mcp_tool.name in READ_ONLY_TOOL_NAMES]
    print(f"Discovered over MCP: {[mcp_tool.name for mcp_tool in discovered_tools]}")
    print(f"Given to the agent : {[mcp_tool.name for mcp_tool in read_only_tools]}\n")

    # --- Build an LLM Agent that Uses MCP Tools ---
    banking_agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=read_only_tools,
        system_prompt=(
            "You are SecureBank India's customer assistant. "
            "Use the provided tools to look up account info. "
            "Always greet customers warmly and respond concisely."
        ),
        name="securebank_agent",
    )
    # Test: agent uses MCP tools to answer a question
    agent_result = await banking_agent.ainvoke({
        "messages": [{"role": "user",
                      "content": "What's the balance for ACC001? Also, who is the account holder?"}]
    })

# Extract final response
final_message = agent_result["messages"][-1]
print("Banking Agent Response (uses MCP tools under the hood):")
print("=" * 60)
print(final_message.content)

Discovered over MCP: ['check_balance', 'transfer_funds', 'get_account_holder']
Given to the agent : ['check_balance', 'get_account_holder']



Banking Agent Response (uses MCP tools under the hood):
The balance for account ACC001 is ₹245,000.00. The account holder is Rajesh Sharma from Mumbai. How else may I assist you?


## 10. A2A: Agent-to-Agent Protocol

**A2A** (launched by Google in April 2025, now stewarded by Linux Foundation) addresses a different problem from MCP:

| Protocol | Solves |
|----------|--------|
| **MCP** | Agent ↔ Tools/Data (one agent uses many tools) |
| **A2A** | Agent ↔ Agent (multiple agents collaborate) |

Built on existing standards: HTTP, Server-Sent Events (SSE), JSON-RPC. Designed for **enterprise**: long-running tasks (seconds to days), authentication, audit trails.

Key concepts:
- **Agent Cards**: JSON documents that advertise an agent's capabilities (like a business card)
- **Tasks**: Units of work that one agent delegates to another
- **Artifacts**: Documents/data exchanged during task execution
- **Status updates**: Real-time progress on long-running tasks

### Agent Card Anatomy

An Agent Card is a JSON document at a well-known URL (e.g., `https://insurance.example.com/.well-known/agent-card.json`) that describes the agent. At its core a card names the agent (`name`), says where to reach it (`url`) and which protocol features it supports (`capabilities`); the rest describes it:
- **Name & description** — What the agent does
- **Skills** — Which tasks it can handle (`skills`), and protocol features such as streaming (`capabilities`)
- **Authentication** — How to authenticate (API key, OAuth, etc. — declared under `securitySchemes` / `security`)
- **Endpoint** — Where to send requests (`url`)

In [11]:
# --- Define Agent Cards (Insurance + Compliance) ---
# In the A2A specification's field names: a card lists SKILLS, `capabilities` holds protocol features,
# and authentication is declared under `securitySchemes` / `security`. Domains use the reserved .example TLD.

INSURANCE_AGENT_CARD = {
    "name": "InsuranceClaimsAgent",
    "description": "Processes insurance claims for SecureLife India. Verifies documents, calculates "
                   "payouts, escalates suspicious claims.",
    "url": "https://agents.securelife.example/a2a/claims",
    "version": "1.2.0",
    "provider": {"organization": "SecureLife India", "url": "https://securelife.example"},
    "capabilities": {"streaming": True, "pushNotifications": False},
    "defaultInputModes": ["application/json"],
    "defaultOutputModes": ["application/json"],
    "securitySchemes": {"bearer": {"type": "http", "scheme": "bearer"}},
    "security": [{"bearer": []}],
    "skills": [
        {"id": "verify_documents", "name": "Verify documents",
         "description": "Verify completeness of claim documents (KYC, bills, prescriptions)",
         "tags": ["claims", "documents"]},
        {"id": "calculate_payout", "name": "Calculate payout",
         "description": "Calculate the eligible payout amount based on policy terms",
         "tags": ["claims", "payout"]},
    ],
}

COMPLIANCE_AGENT_CARD = {
    "name": "ComplianceReviewAgent",
    "description": "Reviews insurance claims for IRDAI regulatory compliance. Checks documentation "
                   "completeness and audit trails.",
    "url": "https://agents.securelife.example/a2a/compliance",
    "version": "2.0.1",
    "provider": {"organization": "SecureLife India", "url": "https://securelife.example"},
    "capabilities": {"streaming": True, "pushNotifications": True},
    "defaultInputModes": ["application/json"],
    "defaultOutputModes": ["application/json"],
    "securitySchemes": {"oauth": {"type": "oauth2", "flows": {"clientCredentials": {
        "tokenUrl": "https://auth.securelife.example/oauth/token",
        "scopes": {"compliance:read": "Read claims and findings",
                   "compliance:write": "Record review outcomes"}}}}},
    "security": [{"oauth": ["compliance:read", "compliance:write"]}],
    "skills": [
        {"id": "validate_claim", "name": "Validate claim",
         "description": "Validate claim against IRDAI Regulation 14 and internal policy CP-007",
         "tags": ["compliance", "claims"]},
        {"id": "flag_anomalies", "name": "Flag anomalies",
         "description": "Flag anomalies (unusual amounts, repeated claims, jurisdictional issues)",
         "tags": ["compliance", "fraud"]},
    ],
}

print("Two A2A Agent Cards defined:")
print(f"  1. {INSURANCE_AGENT_CARD['name']}: {len(INSURANCE_AGENT_CARD['skills'])} skills")
print(f"  2. {COMPLIANCE_AGENT_CARD['name']}: {len(COMPLIANCE_AGENT_CARD['skills'])} skills")

Two A2A Agent Cards defined:
  1. InsuranceClaimsAgent: 2 skills
  2. ComplianceReviewAgent: 2 skills


In [12]:
# --- Pretty-Print One Agent Card ---
print("Insurance Agent Card (full JSON):")
print("=" * 60)
print(json.dumps(INSURANCE_AGENT_CARD, indent=2))

Insurance Agent Card (full JSON):
{
  "name": "InsuranceClaimsAgent",
  "description": "Processes insurance claims for SecureLife India. Verifies documents, calculates payouts, escalates suspicious claims.",
  "url": "https://agents.securelife.example/a2a/claims",
  "version": "1.2.0",
  "provider": {
    "organization": "SecureLife India",
    "url": "https://securelife.example"
  },
  "capabilities": {
    "streaming": true,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "application/json"
  ],
  "defaultOutputModes": [
    "application/json"
  ],
  "securitySchemes": {
    "bearer": {
      "type": "http",
      "scheme": "bearer"
    }
  },
  "security": [
    {
      "bearer": []
    }
  ],
  "skills": [
    {
      "id": "verify_documents",
      "name": "Verify documents",
      "description": "Verify completeness of claim documents (KYC, bills, prescriptions)",
      "tags": [
        "claims",
        "documents"
      ]
    },
    {
      "id": "calculate_pa

### Publish a card, then discover it — for real

Discovery is plain HTTP: a client fetches `/.well-known/agent-card.json` from the agent's host. Any web
server can publish a card, so the cell below writes the Compliance card to a folder and serves it from a
throwaway local web server — then does exactly what a remote agent would: fetch it, check its core fields,
and pick a skill by tag.

In [13]:
import functools
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

CARD_SITE_FOLDER = Path("a2a_card_site")
CARD_SITE_PORT = 8766
(CARD_SITE_FOLDER / ".well-known").mkdir(parents=True, exist_ok=True)
(CARD_SITE_FOLDER / ".well-known" / "agent-card.json").write_text(
    json.dumps(COMPLIANCE_AGENT_CARD, indent=2), encoding="utf-8")


class QuietCardHandler(SimpleHTTPRequestHandler):
    """Serve the card folder without logging every request into the notebook."""

    def log_message(self, message_format, *message_arguments):
        """Silence the default per-request log line."""


card_site = ThreadingHTTPServer(("127.0.0.1", CARD_SITE_PORT),
                                functools.partial(QuietCardHandler, directory=str(CARD_SITE_FOLDER)))
threading.Thread(target=card_site.serve_forever, daemon=True).start()

# --- A remote agent's view: discover, validate, choose a skill ---
card_address = f"http://127.0.0.1:{CARD_SITE_PORT}/.well-known/agent-card.json"
async with httpx.AsyncClient(timeout=5.0) as discovery_client:
    discovered_card = (await discovery_client.get(card_address)).json()
card_site.shutdown()

missing_fields = [field for field in ("name", "url", "capabilities") if field not in discovered_card]
print(f"Fetched {card_address}")
print(f"  agent        : {discovered_card['name']} v{discovered_card['version']}")
print(f"  core fields  : {'all present' if not missing_fields else f'MISSING {missing_fields}'}")
print(f"  endpoint     : {discovered_card['url']}")
print(f"  capabilities : {discovered_card['capabilities']}")
print(f"  authentication: {list(discovered_card['securitySchemes'])} with scopes "
      f"{discovered_card['security'][0]['oauth']}")
fraud_skills = [skill['id'] for skill in discovered_card["skills"] if "fraud" in skill["tags"]]
print(f"  skill for a fraud check: {fraud_skills}")

Fetched http://127.0.0.1:8766/.well-known/agent-card.json
  agent        : ComplianceReviewAgent v2.0.1
  core fields  : all present
  endpoint     : https://agents.securelife.example/a2a/compliance
  capabilities : {'streaming': True, 'pushNotifications': True}
  authentication: ['oauth'] with scopes ['compliance:read', 'compliance:write']
  skill for a fraud check: ['flag_anomalies']


### How A2A Communication Works (Conceptually)

**Live A2A communication requires HTTP servers** — you just ran one to publish a card. Here is the conceptual flow that follows discovery, in the specification's message shapes.

```
Step 1: Discovery
  Client fetches Insurance Agent Card from /.well-known/agent-card.json

Step 2: Task Creation
  Insurance Agent → Compliance Agent:
    POST <card url>  JSON-RPC "message/send": a message whose parts ask for validate_claim {...}
  Response: a Task { "id": "task-123", "status": { "state": "submitted" } }

Step 3: Status Updates (SSE stream)
  Compliance Agent streams: { "id": "task-123", "status": { "state": "working" } }

Step 4: Artifact Delivery
  Final result: { "id": "task-123", "status": { "state": "completed" },
                  "artifacts": [ { "parts": [ { "data": { "compliant": true, "findings": [...] } } ] } ] }
```

> **Why no live task here.** Running a second agent *server* adds nothing the card and the flow above do not
> already show, and the specification is still moving (A2A 1.0 renames `message/send` to `SendMessage`, among
> others). When you build one for real, use the official Python SDK — `pip install a2a-sdk` — and check which
> specification version it targets.

## 11. When to Use MCP vs A2A

| Scenario | Protocol | Why |
|----------|----------|-----|
| Agent needs to call a database/API/tool | **MCP** | Tool integration is what MCP solves |
| Two agents (different teams/vendors) need to collaborate | **A2A** | Agent-level interop with auth |
| Multi-step workflow within one agent system | Custom (or LangGraph) | Both protocols are overkill |
| Agent needs structured data from an internal service | **MCP** | Resources are designed for this |
| Long-running task delegated to specialist agent | **A2A** | Built-in task lifecycle, status updates |
| Multiple agents share one set of tools | **MCP** server, all agents are clients | Reuse, single source of truth |

**Common combination:** Insurance Agent uses MCP for its database (claims, policies) + uses A2A to delegate to Compliance Agent. **MCP for tools, A2A for collaboration.**

## Combined Architecture: MCP + A2A

Real enterprise systems use BOTH protocols:

```
[Insurance Agent]                          [Compliance Agent]
    |                                          |
    | calls tools via MCP                      | calls tools via MCP
    v                                          v
[Insurance MCP Server]                    [Compliance MCP Server]
 - fetch_claim()                           - check_irdai_rule()
 - calculate_payout()                      - flag_anomaly()
    |                                          ^
    |                                          |
    +-------- A2A: validate_claim task --------+
         (agent-to-agent communication)
```

In production the two meet exactly like this: each agent reaches its own tools through MCP, and agents hand work to each other through A2A — the orchestration patterns from Module 7 decide *when*.

## Real-World MCP & A2A Applications

**MCP in Production (April 2026):**
- **Claude Desktop:** Native MCP support for filesystem, GitHub, Slack
- **Cursor / VS Code:** MCP extensions for codebase context
- **Anthropic Apps:** Custom MCP servers for enterprise data
- **150+ orgs supporting** the standard

**A2A in Production:**
- **Google Cloud:** Multi-agent orchestration via A2A
- **Salesforce + SAP:** Cross-vendor agent collaboration
- **Linux Foundation stewardship** since June 2025
- Enterprise focus: auth, audit, long-running tasks

## 12. Stop the server

A background server outlives the cell that started it — and a restarted kernel will not stop it for you.
Stop it explicitly, then prove the port is free.

In [14]:
if http_server_process is not None:
    http_server_process.terminate()
    http_server_process.wait(timeout=10)
    server_log.close()
still_healthy, _ = await wait_until_healthy(timeout=3)
print(f"Server stopped: {not still_healthy}  (port {SERVER_PORT} {'still answering!' if still_healthy else 'is free'})")

Server stopped: True  (port 8765 is free)


## Conclusion

We covered both open protocols for agent interop:
- **MCP:** Built a Banking MCP server with 3 tools, 1 resource, 1 prompt; reached it in memory, over stdio and over Streamable HTTP with the same client code; handed its read-only tools to a LangChain agent
- **A2A:** Defined Agent Cards for Insurance + Compliance, published one and discovered it over HTTP
- **Decision matrix:** When to use MCP (tools) vs A2A (agents)
- **Combined architecture:** Real systems use both — MCP for tools, A2A for agent collaboration

### Next
**The capstone — Compliance Report Generator.** Two weeks, take-home: a LangGraph orchestrator-worker system built on the patterns from Module 7, traced in LangSmith. Start with `CAPSTONE_BRIEF.md` in the capstone folder.

> ⏭️ **After the session.** This section is not run live in the workshop. Work through it afterwards, top to bottom.

## 13. Try on your own

1. **Add a tool, change no client code.** Add `list_recent_transactions(account_id)` to the server file,
   re-run §3, then §4 — discovery finds it without a single edit to the client.
2. **Break the server on purpose.** Start the HTTP server, stop it mid-way, and call a tool. Read the error:
   what does a client see when the far end disappears?
3. **Use it from a desktop assistant.** Point an MCP-capable app (Claude Desktop, VS Code) at
   `banking_mcp_server.py` over stdio. The same file, a different host.
4. **Choose an agent by skill.** Publish the Insurance card too, and write `find_agent(tag)` that fetches
   both cards and returns the `url` of the agent whose skills carry that tag.